In [3]:
import os
from openai import OpenAI
from agents import Agent, function_tool, SQLiteSession, Agent, Runner, trace, ModelSettings
from dotenv import load_dotenv
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import smtplib
import asyncio
from email.message import EmailMessage

load_dotenv(override=True)
MODEL_NAME = 'gpt-5.4-nano'

In [7]:
USER_EMAIL = os.getenv("GOOGLE_USER")
USER_PASS = os.getenv("GOOGLE_PASS")
EMAIL_SERVER = os.getenv("GOOGLE_SMTP")

In [ ]:
USER_EMAIL, USER_PASS, EMAIL_SERVER

In [9]:
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg['From'] = USER_EMAIL
    msg['To'] = USER_EMAIL
    msg['Subject'] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SERVER, 587) as server:
        server.starttls()
        server.login(USER_EMAIL, USER_PASS)
        server.send_message(msg)

In [10]:
send_email("Testing push email", "Hope email gets delivered!!!!", "<html><body><strong>Fingers</strong> crossed....</body></html>")

In [1]:
intro = """
    You are a sales agent working for Netshield,
    A company that provides SaaS tool for early fault prediction in 4G, 5G Sites, this application is powered by AI.
    You write emails.
"""

instruction1 = intro + "Your email style is professional, serious with gravitas and credibility."
instruction2 = intro + "Your email style is witty, engaging and humorous."
instruction3 = intro + "Your email style is concise, to the point, in the style busy senior executives do."

In [4]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instruction1, model=MODEL_NAME)
sales_agent2 = Agent(name="Witty Sales Agent", instructions=instruction2, model=MODEL_NAME)
sales_agent3 = Agent(name="Senior Sales Agent", instructions=instruction3, model=MODEL_NAME)

In [6]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email and for [Name] placeholder use Anonymus")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Early fault prediction for your 4G/5G network — with AI

Hi Anonymus,  

Unexpected 4G/5G faults are costly—downtime, SLA penalties, and wasted field time. Netshield helps operators detect and predict faults early using AI, so teams can act before issues impact customers.  

If you’re open to it, I’d like to share how Netshield identifies emerging faults from site/network signals and what teams typically see in reduced outages and faster response.  

Would you be open to a quick 15-minute call next week? If helpful, reply with your preferred time window and I’ll send a calendar invite.  

Best regards,  
[Your Name]  
Netshield | AI-Powered Early Fault Prediction  
[Phone] | [Email] | [Website]

In [7]:
result2 = Runner.run_streamed(sales_agent2, input="Write a witty email and in [Name] placeholder use Anonymus")
async for event in result2.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Quick question, [Name] — did your network just “ghost” you? 😄

Hi [Name] (aka Anonymous),

Hope your day’s going better than a dropped 5G call during prime time.  
At Netshield, we help detect early faults in 4G/5G sites using AI—so issues don’t get the chance to become full-blown “mystery outages.”

If you’re open to it, I can share how teams use our SaaS to spot problems earlier, reduce downtime, and stop troubleshooting based on vibes.

Want a quick chat this week?  
(Anonymous or not, I promise not to judge your alarm history.)

Best,  
[Your Name]  
Netshield  
AI-Powered Early Fault Prediction for 4G/5G Sites

In [8]:
result3 = Runner.run_streamed(sales_agent3, input="Write professional concise email")
async for event in result3.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Quick question about early fault prediction for your 4G/5G network

Hi [Name],  

I’m [Your Name] from Netshield. We use AI to predict faults early for 4G/5G sites, helping teams reduce downtime and speed up resolution.  

Do you have 15 minutes this week to see if it fits your network ops priorities?  

Best regards,  
[Your Name]  
[Title] | Netshield  
[Phone] | [Email] | [Website]

In [11]:
message = "Write a cold sales email"

with trace("Prallel cold emails"):
    result = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

output = [res.final_output for res in result]
for op in output:
    print(op + "\n\n")

Subject: Reduce downtime risk in your 4G/5G sites with AI early-fault prediction

Hi {{FirstName}},  

I’m {{YourName}} from Netshield. We provide an AI-powered SaaS platform that predicts faults early across 4G/5G sites—helping operators move from reactive troubleshooting to proactive maintenance.

If you’re responsible for site reliability or network uptime, Netshield can help you:  
- Detect fault patterns earlier than traditional alarms  
- Prioritize the highest-risk sites and alarms  
- Reduce repeat incidents and avoidable downtime  

Would you be open to a quick 15-minute call next week to see whether early fault prediction aligns with your current reliability workflow?

Best regards,  
{{YourName}}  
{{Title}} | Netshield  
{{Phone}} | {{Website}}  
{{Email}}


Subject: Quick question—are your 4G/5G sites pre-silent… or secretly failing? 👀

Hi {{Name}},  

I’m {{YourName}} from Netshield. We use AI to predict early faults in 4G/5G sites—so issues don’t wait until the moment ev

In [18]:
decision = """
    You pick the best sales agent email for NettShield from the given options.
    Imagine you're a customer and pick the one you're most likely to respond to.
    Do, not add an explanation, reply with the selected email only.
""" 
sales_picker = Agent(name="Sales Email Picker", instructions=decision, model=MODEL_NAME)

In [19]:
message = "Write a professional NetShield email"
with trace("Sales email writer"):
    res = await asyncio.gather(
        Runner.run(sales_agent1, input=message),
        Runner.run(sales_agent2, input=message),
        Runner.run(sales_agent3, input=message)
    )
    outputs = [result.final_output for result in res]
    emails = "NetShield sales email. \n\n + Email: \n\n" +",".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"The bes one that is selected is :\n\n {best}")

The bes one that is selected is :

 RunResult:
- Last agent: Agent(name="Sales Email Picker", ...)
- Final output (str):
    Subject: Catch network faults before they catch you 😄
    
    Hi [Name],  
    
    Quick question: how often do you hear about an issue *after* it’s already made customers unhappy? With NetShield, you can spot early fault signals in your 4G/5G sites—powered by AI—so problems don’t quietly roam your network like uninvited gremlins.
    
    **NetShield helps you:**
    - Predict faults early using AI-driven insights  
    - Reduce downtime with faster, smarter intervention  
    - Improve network reliability across 4G/5G sites  
    
    If you’re open to it, I’d love to share how NetShield fits your environment and what results teams typically see. Would a quick 15-minute chat next week work?
    
    Best regards,  
    [Your Name]  
    NetShield | AI-Powered Early Fault Prediction  
    [Phone] | [Email] | [Website]
- 1 new item(s)
- 1 raw response(s)
- 0 in